In [1]:
import random
random.seed(42)
import time
import numpy as np
np.random.seed(42)
import h5py
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style('white')
sns.set_style('ticks')
#import bcolz
import pandas
import allel; print('scikit-allel', allel.__version__)
%reload_ext memory_profiler


scikit-allel 1.3.5


In [2]:
callset=allel.read_vcf("/Users/rainlam/Project/3.SV/pca_result/test.recode.vcf")

In [4]:
callset = allel.read_vcf("/Users/rainlam/QUL/QUL.LG1.vcf.gz")

/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/allel/io/vcf_read.py:1732: UserWarning: invalid INFO header: '##INFO=<ID=VDB,Number=1,Type=Float,Description="Variant Distance Bias for filtering splice-site artefacts in RNA-seq data (bigger is better)",Version="3">\n'
  warnings.warn('invalid INFO header: %r' % header)


In [5]:
sorted(callset.keys())

['calldata/GT',
 'samples',
 'variants/ALT',
 'variants/CHROM',
 'variants/FILTER_PASS',
 'variants/ID',
 'variants/POS',
 'variants/QUAL',
 'variants/REF']

In [6]:
chrom = 'LR880645.1'

In [5]:
# population data 
pop = pandas.read_csv("/Users/rainlam/Project/3.SV/pca_result/pop.txt", sep='\t')

In [8]:
allel.GenotypeChunkedArray

allel.model.chunked.GenotypeChunkedArray

In [9]:
g = allel.GenotypeChunkedArray(callset['calldata/GT'])

In [10]:
ac = g.count_alleles()[:]

In [11]:
g

<GenotypeChunkedArray shape=(475067, 20, 2) dtype=int8
   nbytes=18.1M
   values=numpy.ndarray>

In [12]:
# multiallelic
# np.count_nonzero(ac.max_allele() > 1)
# biallelic singletons
np.count_nonzero((ac.max_allele() == 1) & ac.is_singleton(1))

10124

In [13]:
flt = (ac.max_allele() == 1) & (ac[:, :2].min(axis=1) > 1)
gf = g.compress(flt, axis=0)
gf
gn = gf.to_n_alt()
gn

<ChunkedArrayWrapper shape=(112620, 20) dtype=int8 chunks=(52428, 20)
   nbytes=2.1M cbytes=312.2K cratio=7.0
   compression=gzip compression_opts=1
   values=h5py._hl.dataset.Dataset>

In [14]:
def plot_ld(gn, title):
    m = allel.rogers_huff_r(gn) ** 2
    ax = allel.plot_pairwise_ld(m)
    ax.set_title(title)

In [15]:
n = 100000  # number of SNPs to choose randomly
vidx = np.random.choice(gn.shape[0], n, replace=False)
vidx.sort()
gnr = gn.take(vidx, axis=0)
gnr

<ChunkedArrayWrapper shape=(100000, 20) dtype=int8 chunks=(52428, 20)
   nbytes=1.9M cbytes=275.5K cratio=7.1
   compression=gzip compression_opts=1
   values=h5py._hl.dataset.Dataset>